To run this, press "*Runtime*" and press "*Run all*" on a **Google Colab** GPU instance.
<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
</div>

This notebook is adapted for a short-step **training loss evaluation** with Qwen2.5 3B on our 100k balanced LunarLander dataset.
- Model: `Qwen/Qwen2.5-3B`
- Dataset: `Ali2023kosemen/lunarlander_3_layer`
- Goal: short SFT run, inspect train/eval loss, and evaluate behavior without saving the model


## Qwen2.5 3B LunarLander Loss Evaluation

Bu notebook, `Ali2023kosemen/lunarlander_3_layer` veri seti ile Qwen2.5 3B modelinde kisa adimlarda training loss testi yapmak icin hazirlandi.

Dataset linki: https://huggingface.co/datasets/Ali2023kosemen/lunarlander_3_layer
Model linki: https://huggingface.co/Qwen/Qwen2.5-3B


In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

* We support Llama, Mistral, Phi-3, Gemma, Yi, DeepSeek, Qwen, TinyLlama, Vicuna, Open Hermes etc
* We support 16bit LoRA or 4bit QLoRA. Both 2x faster.
* `max_seq_length` can be set to anything, since we do automatic RoPE Scaling via [kaiokendev's](https://kaiokendev.github.io/til) method.
* [**NEW**] We make Gemma-2 9b / 27b **2x faster**! See our [Gemma-2 9b notebook](https://colab.research.google.com/drive/1vIrqH5uYDQwsJ4-OO3DErvuv4pBgVwk4?usp=sharing)
* [**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)
* [**NEW**] We make Mistral NeMo 12B 2x faster and fit in under 12GB of VRAM! [Mistral NeMo notebook](https://colab.research.google.com/drive/17d3U-CAIwzmbDRqbZ9NnpHxCkmXB6LZ0?usp=sharing)

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # LunarLander prompts are short, so 1024 is enough.
dtype = None
load_in_4bit = True # 3B model icin Colab'de 4bit QLoRA tercih ediyoruz.

fourbit_models = [
    "unsloth/Qwen2.5-0.5B-bnb-4bit",
    "unsloth/Qwen2.5-1.5B-bnb-4bit",
    "unsloth/Qwen2.5-3B-bnb-4bit",
    "unsloth/Qwen2.5-7B-bnb-4bit",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


<a name="Data"></a>
### Data Prep
We use the same 100k balanced LunarLander dataset from Hugging Face Hub: [`Ali2023kosemen/lunarlander_3_layer`](https://huggingface.co/datasets/Ali2023kosemen/lunarlander_3_layer).

For loss evaluation, we split the same dataset into train and eval subsets.
We still keep the ShareGPT-style `conversations` field and convert it into a Qwen-compatible ChatML template.


In [4]:
EOS_TOKEN = tokenizer.eos_token

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)

def formatting_prompts_func(examples):
    conversations = examples["conversations"]
    texts = []
    for convo in conversations:
        text = tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

from datasets import load_dataset
raw_dataset = load_dataset("Ali2023kosemen/lunarlander_3_layer", split = "train")
raw_dataset = raw_dataset.shuffle(seed = 3407)
split_dataset = raw_dataset.train_test_split(test_size = 0.02, seed = 3407)

train_dataset = split_dataset["train"].map(formatting_prompts_func, batched = True)
eval_dataset = split_dataset["test"].select(range(min(1000, len(split_dataset["test"]))))
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")
print(train_dataset[0]["text"][:1000])


Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


README.md:   0%|          | 0.00/355 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

{'conversations': [{'from': 'human', 'value': 'State: [x=0.0887, y=-0.0011, vx=-0.0002, vy=-0.0000, angle=-0.0027, angular_vel=0.0000, left_leg=1.0000, right_leg=0.0000]. What action should the lander take?'}, {'from': 'gpt', 'value': 'Action: 0 (do nothing).'}]}


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

<|im_start|>user
State: [x=0.0887, y=-0.0011, vx=-0.0002, vy=-0.0000, angle=-0.0027, angular_vel=0.0000, left_leg=1.0000, right_leg=0.0000]. What action should the lander take?<|im_end|>
<|im_start|>assistant
Action: 0 (do nothing).<|im_end|>
<|endoftext|>


<a name="Train"></a>
### Train the model (short loss evaluation run)
We run a short QLoRA training with evaluation steps enabled so we can inspect both training loss and eval loss.

This notebook is for **loss inspection and quick evaluation**, not for final model export.


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 40,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        evaluation_strategy = "steps",
        eval_steps = 5,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "loss_eval_outputs",
        report_to = "none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [6]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
0.566 GB of memory reserved.


In [7]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,3.002942
2,3.012417
3,2.909940
4,2.706568
5,2.411764
6,2.197814
7,1.894589
8,1.668555
9,1.449527
10,1.279833


In [8]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

75.6407 seconds used for training.
1.26 minutes used for training.
Peak reserved memory = 0.914 GB.
Peak reserved memory for training = 0.348 GB.
Peak reserved memory % of max memory = 6.276 %.
Peak reserved memory for training % of max memory = 2.39 %.


### Loss Curves
After training, we extract the trainer log history and plot both training loss and eval loss to inspect whether the run is learning stably.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
display(history.tail())

train_history = pd.DataFrame()
eval_history = pd.DataFrame()

if "loss" in history.columns:
    train_history = history.dropna(subset=["loss"])[["step", "loss"]].copy()

if "eval_loss" in history.columns:
    eval_history = history.dropna(subset=["eval_loss"])[["step", "eval_loss"]].copy()

plt.figure(figsize=(10, 5))
if not train_history.empty:
    plt.plot(train_history["step"], train_history["loss"], marker="o", label="train loss")
if not eval_history.empty:
    plt.plot(eval_history["step"], eval_history["eval_loss"], marker="s", label="eval loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Qwen2.5 3B - LunarLander short-run loss curves")
plt.grid(alpha=0.3)
if not train_history.empty or not eval_history.empty:
    plt.legend()
plt.show()

if not train_history.empty:
    print(f"Final train loss: {train_history['loss'].iloc[-1]:.4f}")
    print(f"Min train loss: {train_history['loss'].min():.4f}")

if not eval_history.empty:
    print(f"Final eval loss: {eval_history['eval_loss'].iloc[-1]:.4f}")
    print(f"Min eval loss: {eval_history['eval_loss'].min():.4f}")
else:
    print("No eval_loss was logged in this run.")



<a name="Inference"></a>
### Inference
Let's run a quick inference check after the short training run.


In [ ]:
import torch
import warnings
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore", category=FutureWarning)
hf_logging.set_verbosity_error()
model.eval()

def manual_chatml_decode(model, tokenizer, user_prompt, max_new_tokens=32, stream=False):
    prompt_text = tokenizer.apply_chat_template(
        [{"from": "human", "value": user_prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    generated = encoded["input_ids"]
    attention_mask = encoded.get("attention_mask", torch.ones_like(generated))

    if stream:
        print(prompt_text, end="")

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(
                input_ids=generated,
                attention_mask=attention_mask,
                use_cache=False,
            )
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            token_text = tokenizer.decode(next_token[0], skip_special_tokens=False)
            if stream:
                print(token_text, end="", flush=True)

            full_text = tokenizer.decode(generated[0], skip_special_tokens=False)
            if next_token.item() == tokenizer.eos_token_id or "<|im_end|>" in full_text:
                break

    full_text = tokenizer.decode(generated[0], skip_special_tokens=False)
    answer = full_text.split("<|im_start|>assistant\n", 1)[-1]
    answer = answer.split("<|im_end|>", 1)[0].strip()
    return answer, full_text

answer, full_text = manual_chatml_decode(
    model,
    tokenizer,
    "State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?",
    max_new_tokens=32,
)

print("Model answer:")
print(answer)




`TextStreamer` + `model.generate(...)` bazen Unsloth'un hizlandirilmis inference yolunda shape hatasina dusuyor. Asagidaki hucre ayni promptu manuel greedy decoding ile token token yazdirir ve bu problemi dolasir.


In [ ]:
_ = manual_chatml_decode(
    model,
    tokenizer,
    "State: [x=0.0005, y=1.4126, vx=0.0492, vy=0.0739, angle=-0.0006, angular_vel=-0.0112, left_leg=0.0000, right_leg=0.0000]. What action should the lander take?",
    max_new_tokens=32,
    stream=True,
)
print()
